In [1]:
import torch
import transformers
import numpy as np
import wandb
from datasets import load_dataset, Dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model
from typing import List
from tqdm import tqdm
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer

from utils.prompting import *
from utils.utils import write_json, append_jsonl, normalize_answer, set_seed, load_timeqax_data
import fire
wandb.init(mode="disabled")
import pandas as pd

from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

import re
import statistics

2026-01-11 17:35:30.405018: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-11 17:35:30.432959: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-11 17:35:30.432988: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-11 17:35:30.433002: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-11 17:35:30.439188: I tensorflow/core/platform/cpu_feature_g

[2026-01-11 17:35:32,451] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/an/anaconda3/envs/deep_learning_env_22/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/an/anaconda3/envs/deep_learning_env_22/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda-11.8/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/an/anaconda3/envs/deep_learning_env_22/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda-11.8/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/an/anaconda3/envs/deep_learning_env_22/compiler_compat/ld: /usr/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/an/anaconda3/envs/deep_learning_env_22/compiler_compat/ld: /usr/local/cuda-11.8/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/an/anaconda3/envs/deep_learning_env_22/compiler_compat/ld: /usr/local/cuda-11.8/lib64/libcufile.so: undefined reference to `std::ostr

In [2]:
# training hyperparams
batch_size = 1
num_epochs = 1
# learning_rate = 5e-6
learning_rate = 2e-7
# lora hyperparams
# lora_r = 8
# lora_alpha = 16
# lora_r = 16
# lora_alpha = 32
lora_r = 64
lora_alpha = 16
lora_dropout = 0.05
lora_target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj']
train_on_inputs = False  # if False, masks out inputs in loss
add_eos_token = False
eval_steps=200
save_steps=200
save_total_limit=10
seed=201
# debug_mode = False
debug_mode = True

# Read Data

In [48]:
train_df = pd.read_pickle("../data/rl_train/train.pkl")
train_df = train_df.reset_index().rename(columns={'index': 'id'})
train_df['user_profiles'] = train_df['s3_user_profiles']

In [50]:
train_df

,id,category,product_name,product_reviews,user_id,filtered_hist_vote_written,s1_kp,s2_kp_helpfulness_cal,s3_user_profiles,s4_helpful_kps_filtered,s5_annotated_personalized_summaries,user_profiles
0,0,Family,Connect Four,[ Most recently i have had connect four ins...,5013640,"[ I was surfing the net for some presents, ...",[Connect Four is a simple and classic two-play...,[{'kp': 'A compact travel version of Connect F...,### User Profile Summary\n\n**Personality Trai...,[A compact travel version of Connect Four is a...,Connect Four stands out as a practical and eng...,### User Profile Summary\n\n**Personality Trai...
1,1,Family,Johnson's Baby Shampoo,[ Who would have thought I would be using j...,5016911,[ I don t know how i found this magic potio...,"[Johnson's Baby Shampoo is mild and gentle, ma...",[{'kp': 'A small amount of shampoo lathers wel...,### User Profile Summary\n\n**Personality Trai...,[A small amount of shampoo lathers well and co...,If you’re on the hunt for a baby shampoo that’...,### User Profile Summary\n\n**Personality Trai...
2,2,Shopping,Tiny Computers (Shop),[ Having purchased and used a Tiny computer...,5013640,[ The btcellenet pay as you go package is a...,[Tiny Computers' customer service is extremely...,[{'kp': 'Many essential components in Tiny Com...,### User Profile Summary\n\n**Personality Trai...,[Tiny Computers covered repair costs even when...,If you’re on the lookout for a reliable PC wit...,### User Profile Summary\n\n**Personality Trai...
3,3,Entertainment,The Sun,[ Sometimes i don t really think the sun kn...,5001205,[ I think sister sister is one of the best ...,[The Sun is known for sensational headlines an...,[{'kp': 'The Sun frequently trivializes seriou...,### User Profile Summary\n\n**Personality Trai...,[The Sun frequently trivializes serious issues...,"If you’re looking for a quick, easy read that ...",### User Profile Summary\n\n**Personality Trai...
4,4,Entertainment,Channel 4 - Trigger Happy TV,[ Trigger Happy TV is so good that everytim...,5001825,[ At home with the Braithwaites follows the...,[Dom Joly plays hilarious pranks on unsuspecti...,"[{'kp': 'Dom Joly's antics, like pretending to...",### User Profile Summary\n\n**Personality Trai...,"[Dom Joly's antics, like pretending to be a pa...",If you appreciate comedy that thrives on genui...,### User Profile Summary\n\n**Personality Trai...
...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,Beauty,Original Source Tea Tree & Mint Shower Gel,[ I logged on to the Original Sources websi...,19851,"[ As a make up-aholic,I have tried just abo...",[The shower gel provides an invigorating and r...,[{'kp': 'It is not recommended for use on smal...,### User Profile: connoisseur_haggler\n\n**Per...,[It is not recommended for use on small cuts o...,If you’re after a shower gel that wakes you up...,### User Profile: connoisseur_haggler\n\n**Per...
496,496,Entertainment,Popstars,[ Pop stars is such an amazing programme. t...,5013640,[ I don t get this programme.Why would anyo...,[Popstars is a fly-on-the-wall documentary fol...,[{'kp': 'Popstars was praised for revealing th...,### User Profile Summary\n\n**Personality Trai...,[Popstars was praised for revealing the realit...,If you’re curious about the gritty realities b...,### User Profile Summary\n\n**Personality Trai...
497,497,Entertainment,ITV - Blind Date,[ Why can t we have something decent on Sat...,5013640,[ I don t get this programme.Why would anyo...,[Most contestants participate in Blind Date ma...,[{'kp': 'Blind Date is sometimes considered a ...,### User Profile Summary\n\n**Personality Trai...,[Blind Date is sometimes considered a fun and ...,"If you’re curious about ITV’s *Blind Date*, he...",### User Profile Summary\n\n**Personality Trai...
498,498,DVDs,Pretty Woman (DVD),"[ Pretty Woman, the classic tale of opposit...",5259809,[ How many cheesy american teenage films do...,[Pretty Woman is a modern Cinderella story abo...,[{'kp': 'The film is entertaining from begin

In [5]:
train_data = train_df

# Setup

## Policy Model

In [6]:
TEMPLATE = get_prompt("helpfulsumm_cot_helpful_pos")

In [7]:
device_map = "auto"
world_size = int(os.environ.get("WORLD_SIZE", 1))
ddp = world_size != 1
if ddp:
    device_map = {"": int(os.environ.get("LOCAL_RANK") or 0)}

In [8]:
base_model = "hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4"
source_path = '../models/stage_1_helpfulsumm_ft/checkpoint-800'

In [9]:
output_dir = f'../models/stage_2_helpfulsumm_rl_2/'

In [10]:
os.makedirs(output_dir, exist_ok=True)
set_seed(seed=seed)

In [11]:
policy_model = AutoModelForCausalLM.from_pretrained(
    source_path,
    device_map='auto',
    trust_remote_code=False,
    revision="main"
)

/mnt/c/Users/antan/Desktop/PHD READING/Product_Question_Answering/JustiLM_Question_based_Summarization/AutoGPTQ/auto_gptq/nn_modules/triton_utils/kernels.py:411: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, qweight, scales, qzeros, g_idx, bits, maxq):
/mnt/c/Users/antan/Desktop/PHD READING/Product_Question_Answering/JustiLM_Question_based_Summarization/AutoGPTQ/auto_gptq/nn_modules/triton_utils/kernels.py:419: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/mnt/c/Users/antan/Desktop/PHD READING/Product_Question_Answering/JustiLM_Question_based_Summarization/AutoGPTQ/auto_gptq/nn_modules/triton_utils/kernels.py:461: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` i

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
from auto_gptq import exllama_set_max_input_length
policy_model = exllama_set_max_input_length(policy_model, max_input_length=65536)

In [13]:
policy_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (rotary_emb): LlamaRotaryEmbedding()
          (k_proj): lora.QuantLinear(
            (base_layer): QuantLinear()
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=1024, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (quant_linear_module): QuantLinear()
          )
          (o_proj): lora.QuantLinear(
            (base_layer): QuantLinear()
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inpla

In [14]:
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

In [15]:
ref_model = create_reference_model(policy_model)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(ref_model)

In [16]:
policy_model = get_peft_model(policy_model, peft_config)
policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(policy_model)

In [17]:
tokenizer = transformers.AutoTokenizer.from_pretrained(
    base_model,
    padding_side="right",
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = (
    tokenizer.eos_token_id
)
tokenizer.add_prefix_space = False

In [18]:
def tokenize(prompt):
    result = tokenizer(
        prompt
    )
    result["labels"] = result["input_ids"].copy()

    return result

In [19]:
def generate_and_tokenize_prompt(data_point, gen=False, eval=False):
    product_name = data_point['product_name']
    product_reviews = data_point['product_reviews']

    hist_vote_written = data_point['filtered_hist_vote_written']
    
    query = TEMPLATE.format(product_name = product_name, product_reviews = product_reviews, 
                            hist_vote_written = hist_vote_written)
    
    if eval:
        msg = [{"role": "user", "content": query}, {"role": "assistant", "content": response}]
        formatted_prompt =  tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        tokenized_full_prompt = tokenize(formatted_prompt)
    elif gen:
        msg = [{"role": "user", "content": query}]
        formatted_prompt =  tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        tokenized_full_prompt = tokenize(formatted_prompt)
    else:
        msg = [{"role": "user", "content": query}, {"role": "assistant", "content": response}]
        formatted_prompt =  tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        tokenized_full_prompt = tokenize(formatted_prompt)

    if not train_on_inputs:
        user_msg = [{"role": "user", "content": query}]
        user_prompt = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=False)
        tokenized_user_prompt = tokenize(user_prompt)
        user_prompt_len = len(tokenized_user_prompt["input_ids"])
        if add_eos_token:
            user_prompt_len -= 1
        tokenized_full_prompt["labels"] = [-100] * user_prompt_len + tokenized_full_prompt["labels"][user_prompt_len:]

    tokenized_full_prompt['id'] = int(data_point['id'])
    
    return tokenized_full_prompt

def generate_and_tokenize_prompt_gen(data_point):
    return generate_and_tokenize_prompt(data_point, gen=True)

def process_query_tensor(qt):
    i = qt.tolist().index(128000)
    return qt[i:]

def save_check_point():
    pass

def get_step_logs(stats, batch, rewards, step_i):
    logs = {}
    logs['step'] = step_i
    rewards = torch.stack(rewards)
    for k, v in stats.items():
        if not isinstance(v, np.ndarray):
            logs[k] = v
    logs["env/reward_mean"] = torch.mean(rewards).item()	# torch.mean(rewards).cpu().numpy().item()
    logs["env/reward_std"] = torch.std(rewards).item()		# torch.std(rewards).cpu().numpy().item()
    logs["env/reward_dist"] = rewards.tolist()	# rewards.cpu().numpy()
    return logs

def get_text_logs(pred, ans, epoch, step_i, idx_b):
    logs = []
    for p, a, idx in zip(pred, ans, idx_b):
        d = {'epoch': epoch, 'step': step_i, 'id': idx, 'output': p}
        logs.append(d)
        idx += 1
    return logs

In [20]:
train_data

,id,category,product_name,product_reviews,user_id,filtered_hist_vote_written,s1_kp,s2_kp_helpfulness_cal,s3_user_profiles,s4_helpful_kps_filtered,s5_annotated_personalized_summaries
0,0,Family,Connect Four,[ Most recently i have had connect four ins...,5013640,"[ I was surfing the net for some presents, ...",[Connect Four is a simple and classic two-play...,[{'kp': 'A compact travel version of Connect F...,### User Profile Summary\n\n**Personality Trai...,[A compact travel version of Connect Four is a...,Connect Four stands out as a practical and eng...
1,1,Family,Johnson's Baby Shampoo,[ Who would have thought I would be using j...,5016911,[ I don t know how i found this magic potio...,"[Johnson's Baby Shampoo is mild and gentle, ma...",[{'kp': 'A small amount of shampoo lathers wel...,### User Profile Summary\n\n**Personality Trai...,[A small amount of shampoo lathers well and co...,If you’re on the hunt for a baby shampoo that’...
2,2,Shopping,Tiny Computers (Shop),[ Having purchased and used a Tiny computer...,5013640,[ The btcellenet pay as you go package is a...,[Tiny Computers' customer service is extremely...,[{'kp': 'Many essential components in Tiny Com...,### User Profile Summary\n\n**Personality Trai...,[Tiny Computers covered repair costs even when...,If you’re on the lookout for a reliable PC wit...
3,3,Entertainment,The Sun,[ Sometimes i don t really think the sun kn...,5001205,[ I think sister sister is one of the best ...,[The Sun is known for sensational headlines an...,[{'kp': 'The Sun frequently trivializes seriou...,### User Profile Summary\n\n**Personality Trai...,[The Sun frequently trivializes serious issues...,"If you’re looking for a quick, easy read that ..."
4,4,Entertainment,Channel 4 - Trigger Happy TV,[ Trigger Happy TV is so good that everytim...,5001825,[ At home with the Braithwaites follows the...,[Dom Joly plays hilarious pranks on unsuspecti...,"[{'kp': 'Dom Joly's antics, like pretending to...",### User Profile Summary\n\n**Personality Trai...,"[Dom Joly's antics, like pretending to be a pa...",If you appreciate comedy that thrives on genui...
...,...,...,...,...,...,...,...,...,...,...,...
495,495,Beauty,Original Source Tea Tree & Mint Shower Gel,[ I logged on to the Original Sources websi...,19851,"[ As a make up-aholic,I have tried just abo...",[The shower gel provides an invigorating and r...,[{'kp': 'It is not recommended for use on smal...,### User Profile: connoisseur_haggler\n\n**Per...,[It is not recommended for use on small cuts o...,If you’re after a shower gel that wakes you up...
496,496,Entertainment,Popstars,[ Pop stars is such an amazing programme. t...,5013640,[ I don t get this programme.Why would anyo...,[Popstars is a fly-on-the-wall documentary fol...,[{'kp': 'Popstars was praised for revealing th...,### User Profile Summary\n\n**Personality Trai...,[Popstars was praised for revealing the realit...,If you’re curious about the gritty realities b...
497,497,Entertainment,ITV - Blind Date,[ Why can t we have something decent on Sat...,5013640,[ I don t get this programme.Why would anyo...,[Most contestants participate in Blind Date ma...,[{'kp': 'Blind Date is sometimes considered a ...,### User Profile Summary\n\n**Personality Trai...,[Blind Date is sometimes considered a fun and ...,"If you’re curious about ITV’s *Blind Date*, he..."
498,498,DVDs,Pretty Woman (DVD),"[ Pretty Woman, the classic tale of opposit...",5259809,[ How many cheesy american teenage films do...,[Pretty Woman is a modern Cinderella story abo...,[{'kp': 'The film is entertaining from beginni...,### User Profile Summary\n\n**Personality Trai...,[The film is entertaining from beginning to en...,“Pretty Woman” on DVD offers an entertaining r...


In [21]:
print('convert data ...')
train_data = Dataset.from_pandas(train_data).map(generate_and_tokenize_prompt_gen)
train_data = train_data.select_columns(['input_ids', 'attention_mask', 'labels', 'id'])

convert data ...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [22]:
if not ddp and torch.cuda.device_count() > 1:
    # keeps Trainer from trying its own DataParallelism when more than 1 gpu is available
    ref_model.is_parallelizable = True
    ref_model.model_parallel = True
    policy_model.is_parallelizable = True
    policy_model.model_parallel = True

In [23]:
micro_batch_size = 1
_batch_size = micro_batch_size
gradient_accumulation_steps = 1
_batch_size = micro_batch_size
_lr = learning_rate
config = PPOConfig(
    reward_model=None,
    kl_penalty="kl",
    batch_size=2,
    mini_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    ppo_epochs=num_epochs,
    learning_rate=_lr,
    remove_unused_columns=False,
    seed=42,
)

trainer = PPOTrainer(
    config=config,
    tokenizer=tokenizer,
    model=policy_model,
    ref_model=ref_model,
    dataset=train_data,
    data_collator=transformers.DataCollatorForSeq2Seq(
            tokenizer,
            pad_to_multiple_of=8,
            return_tensors="pt",
            padding=True,
        ),
)

generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": False,
    "repetition_penalty": 1.0,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": 1000,
#     "max_new_tokens": 15,
    "eos_token_id": tokenizer.eos_token_id,
}

## Claim Extraction

In [24]:
from openai import OpenAI
client = OpenAI(
    api_key = api_key
)

model="gpt-4.1"
prompt = "Once upon a time"

In [25]:
def get_completion(prompt, model=model):
    messages = [{"role": "user", "content": prompt}]
    ct = 0
    while ct < 2:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=1000,
                temperature=0, # this is the degree of randomness of the model's output
            )
            return response.choices[0].message.content
            break
        except Exception as e:
            print(e)
            ct += 1
        
    return response.choices[0].message.content

In [26]:
base_prompt = get_prompt("summary_kp_extraction")

In [27]:
def claim_extraction_from_summary(text):
    claim_split_prompt = base_prompt %(text)
    attempt = 0
    claim_split_response = None
    while attempt < 3 and claim_split_response == None:
        claim_split_response = get_completion(claim_split_prompt, model)
        attempt += 1
        
    if claim_split_response != None:
        claim_split_response = claim_split_response.replace("`", "").replace("json", "")
        generated_summary_kps = ast.literal_eval(claim_split_response)
    else:
        generated_summary_kps = []
    return generated_summary_kps

## KP Helpfulness Reward Model

In [28]:
import math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig, Trainer, TrainingArguments
from datasets import Dataset
from scipy.stats import pearsonr, spearmanr
import ast

In [29]:
class CrossEncoderRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.device = torch.device("cpu")  # <<< force CPU
        
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config, low_cpu_mem_usage=True)
        self.dropout = nn.Dropout(self.config.hidden_dropout_prob if hasattr(self.config, "hidden_dropout_prob") else 0.1)
        self.head = nn.Linear(self.config.hidden_size, 1)  # scalar
        
         # Ensure whole module is on CPU (paranoia; everything should already be)
        self.to(self.device)
        self.eval()  # inference by default for a reward model
        
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        if input_ids.device.type != "cpu":
            input_ids = input_ids.to("cpu")
        if attention_mask is not None and attention_mask.device.type != "cpu":
            attention_mask = attention_mask.to("cpu")
        if token_type_ids is not None and token_type_ids.device.type != "cpu":
            token_type_ids = token_type_ids.to("cpu")

        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        # pool: prefer CLS token; if pooler exists (BERT), you can use pooler_output
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = out.last_hidden_state[:, 0, :]  # CLS
        x = self.dropout(pooled)
        raw = self.head(x).squeeze(-1)             # shape [B]
        pred01 = torch.sigmoid(raw)                 # in [0,1]
        pred = pred01 * 4.0 + 1.0                  # bound to [1,5]
        outputs = {"logits": pred.unsqueeze(-1)}   # Trainer expects "logits"
        if labels is not None:
            labels = labels.to(pred.dtype)
            loss = F.mse_loss(pred, labels)
            outputs["loss"] = loss
        return outputs
    
reward_model = CrossEncoderRegressor("microsoft/deberta-v2-xlarge")
reward_model.load_state_dict(torch.load('../models/stage_2_helpful_opinion_reward_deberta_ft/model_3_epoch_good.pth', map_location=torch.device("cpu")))

/tmp/ipykernel_8379/386505670.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  reward_model.load_state_dict(torch.load('../models/stage_2_helpful_opinion_reward_deberta_

<All keys matched successfully>

In [30]:
reward_model_checkpoint = "microsoft/deberta-v2-xlarge"   # swap to "bert-base-uncased" if you prefer BERT
rewrd_tokenizer = AutoTokenizer.from_pretrained(reward_model_checkpoint, use_fast=True)

/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [31]:
def tokenize_function(examples):
    out = rewrd_tokenizer(
        examples["kp"],
        examples["history_text"],
        padding="max_length", 
        truncation=True,
        max_length=MAX_LEN,
    )
    out["user_id"] = examples["user_id"]
    return out

In [32]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_rouge_score(row):
    rouge1_scores, rouge2_scores, rougel_scores = [], [], []
    for rev in row['filtered_hist_vote_written']:
        scores = scorer.score(row['key_point'], rev)
        rouge1_scores += [scores['rouge1'].fmeasure]
        rouge2_scores += [scores['rouge2'].fmeasure]
        rougel_scores += [scores['rougeL'].fmeasure]
        
    row['rouge1_scores'] = rouge1_scores
    row['rouge2_scores'] = rouge2_scores
    row['rougel_scores'] = rougel_scores

    return row

In [33]:
def filter_hist_vote_written(row):
    my_df = row[['filtered_hist_vote_written', 'rouge1_scores', 'rouge2_scores', 'rougel_scores']].to_frame().T
    my_df = my_df.explode(['filtered_hist_vote_written', 'rouge1_scores', 'rouge2_scores', 'rougel_scores'])
    my_df = my_df.sort_values(by=['rougel_scores', 'rouge2_scores', 'rouge1_scores'], ascending=False)
    row['sorted_hist_vote_written'] = my_df['filtered_hist_vote_written'].tolist()
    return row

In [34]:
def join_history(reviews):
    # Use tokenizer-specific separator; helps the encoder segment reviews
    sep = rewrd_tokenizer.sep_token if rewrd_tokenizer.sep_token is not None else " "
    return f" {sep} ".join([r.strip() for r in reviews])

## Persona Consistency Reward Model

In [55]:
from openai import OpenAI
client = OpenAI(
    api_key = api_key
)
persona_reward_model="gpt-3.5-turbo"

In [56]:
def get_scoring_completion(prompt, model=persona_reward_model):
    ct = 0
    all_responses = None
    while ct < 2:
        try:
            if ct == 0:
                _response = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": prompt}],
                    temperature=2,
                    max_tokens=150,
                    top_p=1,
                    frequency_penalty=0,
                    presence_penalty=0,
                    stop=None,
                    n=5
                )

                all_responses = [_response.choices[i].message.content for i in
                                 range(len(_response.choices))]
            else:
                _response = personal_client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": prompt}],
                    temperature=2,
                    max_tokens=150,
                    top_p=1,
                    frequency_penalty=0,
                    presence_penalty=0,
                    stop=None,
                    n=5
                )
                all_responses = [_response.choices[i].message.content for i in
                                 range(len(_response.choices))]
        
            break
        except Exception as e:
            print(e)
                
    return all_responses

In [57]:
# BEST (OK) BEST
persona_reward_prompt = get_prompt("persona_alignment_reward_scoring")

In [58]:
def calculate_persona_reward(df):
    product_name = df.iloc[0]['product_name']
    ext = re.findall(r"# Personalized Summary:\n*((?:.+\n*)+)", df.iloc[0]['generated_personalized_summaries'])
    if len(ext) > 0:
        generated_personalized_summary = ext[0]
    else:
        generated_personalized_summary = df.iloc[0]['generated_personalized_summaries']
    hist_vote_written = [rev.strip() for rev in df.iloc[0]['hist_vote_written']]
    user_profile = df.iloc[0]['user_profiles']
    
    prompt = persona_reward_prompt % (product_name, generated_personalized_summary, user_profile)
    
    all_responses = get_scoring_completion(prompt)
    
    rating_extractions =  [[int(rating) for rating in re.findall(r'[0-9]+', ext_response)[:6] if 1 <= int(rating) <= 5] for ext_response in all_responses]
    rating_extractions = [run for run in rating_extractions if len(run) == 6]
    
    if len(rating_extractions) == 0:
        personal_utterance_reward = 1
    else:
        personal_utterance_reward = statistics.mean([statistics.mean(run) for run in rating_extractions])
    
    return personal_utterance_reward, rating_extractions, all_responses

# Train

## Setup

In [59]:
def calculate_kp_helpfulness_score(entry_id, generated_summary, generated_summary_kps):
    kp_df = pd.DataFrame({'id': [entry_id for i in range(len(generated_summary_kps))] , 'key_point': generated_summary_kps})
    kp_df['generated_personalized_summaries'] = generated_summary
    kp_df = kp_df.merge(train_df[['id', 'product_name', 'category', 'user_id']])
    
    kp_df = kp_df.merge(train_df)
    kp_df = kp_df.apply(calculate_rouge_score, axis=1)
    kp_df = kp_df.apply(filter_hist_vote_written, axis=1)
    kp_df = kp_df[['id', 'category', 'product_name', 'user_id', 'user_profiles', 'key_point', 'sorted_hist_vote_written']].\
        rename(columns={'sorted_hist_vote_written': 'hist_vote_written'})
    kp_df['history_text'] = kp_df['hist_vote_written'].apply(join_history)
    kp_df = kp_df.rename(columns={'key_point': 'kp', 'user_id': 'user_id'})
    
    kp_data = Dataset.from_pandas(kp_df)
    tokenized_kp_data = kp_data.map(tokenize_function, batched=True, remove_columns=kp_data.column_names)
    tokenized_kp_data.set_format("torch")
    from torch.utils.data import DataLoader
    eval_dataloader = DataLoader(tokenized_kp_data, batch_size=2)
    
    reward_model.eval()
    output = []
    for reward_batch in eval_dataloader:
    #     batch_inputs, batch_masks, _ = tuple(b.to(device) for b in batch)
        batch_inputs = reward_batch['input_ids'].to("cpu")
        batch_masks = reward_batch['attention_mask'].to("cpu")
        with torch.no_grad():
            output += reward_model(batch_inputs, batch_masks)["logits"].view(1,-1).tolist()[0]
    
    kp_df['predicted_kp_helpfulness'] = output
    
    return kp_df

In [60]:
def save_rl_data(root_path, kp_df):
    os.makedirs(root_path, exist_ok=True)
    category = kp_df['category'].iloc[0]
    user_id = kp_df['user_id'].iloc[0]
    idx = str(kp_df['id'].iloc[0])
    kp_df.to_pickle(root_path + f"/{category}_{idx}_{user_id}.pkl")

## Train

In [61]:
from datasets.utils.logging import disable_progress_bar
disable_progress_bar()

In [62]:
epochs = num_epochs
print("TOTAL EPOCHS: ", num_epochs)
step_i = 0

TOTAL EPOCHS:  1


In [63]:
MAX_LEN = 512

In [64]:
root_path = output_dir + "rl_output/"

In [65]:
responses = []

In [66]:
import gc
import torch

In [132]:
for epoch in tqdm(range(epochs), "epoch:", ncols=100):
    for batch in tqdm(trainer.dataloader, desc=f"Batch (Epoch {epoch + 1})", leave=False, ncols=100):
        query_tensors = batch["input_ids"]
        input_tensors_b = [process_query_tensor(qt) for qt in query_tensors]
        response_tensors_b = []
        for input_tensors in input_tensors_b:
            response_tensors = trainer.generate([input_tensors], return_prompt=False, **generation_kwargs)
            response_tensors_b += response_tensors
        
        response_b = [tokenizer.decode(rt, skip_special_tokens=True).strip() for rt in response_tensors_b]
        responses += response_b
        batch['response'] = response_b
        id_b = batch['id'].tolist()

        print(f'step-{step_i} >>>')
        
        reward_b = []
        for summ in response_b:
            if len(summ.strip()) > 0:
                ############################### KP HELPFULNESS REWARD
                generated_summary_kps = claim_extraction_from_summary(summ)
                if len(generated_summary_kps) > 0:
                    kp_df = calculate_kp_helpfulness_score(id_b[0], summ, generated_summary_kps)
                    kp_helpfulness_reward = kp_df['predicted_kp_helpfulness'].mean()
                    kp_df['generated_personalized_summaries'] = summ
                

                    ############################### PERSONA CONSISTENCY REWARD
                    personal_utterance_reward, rating_extractions, all_responses = calculate_persona_reward(kp_df)

                    ############################### SAVE
                    kp_df['kp_helpfulness_reward'] = kp_helpfulness_reward
                    kp_df['personal_utterance_rating_extractions'] = [rating_extractions for i in range(len(kp_df))]
                    kp_df['personal_utterance_all_responses'] = [all_responses for i in range(len(kp_df))]
                    kp_df['personal_utterance_reward'] = personal_utterance_reward
                    save_rl_data(root_path, kp_df)
                    
                else:
                    print("############################ NOTE ############################")
                    kp_df = 123
                    kp_helpfulness_reward = 0
                    personal_utterance_reward, rating_extractions, all_responses = calculate_persona_reward(kp_df)

                ############################### COMBINE
                final_reward = kp_helpfulness_reward * 0.5 + personal_utterance_reward * 0.5
                reward_b += [torch.tensor(final_reward)]
            else:
                reward_b += [torch.tensor(0)]
        
        stats = trainer.step([qt for qt in query_tensors], [rt for rt in response_tensors_b], reward_b)

        step_logs = get_step_logs(stats, batch, reward_b, step_i)
        append_jsonl(step_logs, os.path.join(output_dir, 'logs.jsonl'))

        if step_i % 50 == 0:
            checkpoint_folder_name = f"step-{step_i}"
            checkpoint_dir = os.path.join(output_dir, checkpoint_folder_name)
            os.makedirs(checkpoint_dir)
            trainer.model.save_pretrained(checkpoint_dir, safe_serialization=False)
            step_stats = {k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in stats.items()}
            write_json(step_stats, os.path.join(checkpoint_dir, 'reward_stats.json'))

        step_i += 1
        del query_tensors, input_tensors_b, response_tensors_b, response_tensors
        del kp_helpfulness_reward
        del personal_utterance_reward, rating_extractions, all_responses
        del reward_b, kp_df
        gc.collect()
        torch.cuda.empty_cache()  # harmless; can help reduce fragmentation

    if step_i > 150000:
        break

Batch (Epoch 1):   0%|                                                      | 0/250 [00:00<?, ?it/s]You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn

step-0 >>>


/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/peft/utils/save_and_load.py:154: UserWarning: Could not find a config file in ./model/Meta-Llama-3.1-8B-Instruct-GPTQ-INT4 - will assume that the vocabulary was not modified.
  warnings.warn(


>>> write to saved/lora_llama318ft_policy_r8_lr2e-07_b1_sd201_general_sample_500_new_filtered_full_rewards_great/step-0/reward_stats.json



Batch (Epoch 1):   0%|▏                                           | 1/250 [01:21<5:37:39, 81.36s/it]/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


...


Batch (Epoch 1):  99%|████████████████████████████████████████▋| 248/250 [7:00:49<03:36, 108.45s/it]/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


step-248 >>>



Batch (Epoch 1): 100%|████████████████████████████████████████▊| 249/250 [7:03:00<01:55, 115.18s/it]/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/an/anaconda3/envs/deep_learning_env_22/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


step-249 >>>



epoch:: 100%|████████████████████████████████████████████████████| 1/1 [7:04:55<00:00, 25495.55s/it]
